# company_events — 다가오는 일정 모으기(2026-11-23)

홈 '한 화면에 보기'의 **다가오는 일정** 칸에 쓸 일정을 공식 출처에서 모아 DB 표 `news.company_events`에 넣는다. LLM은 쓰지 않는다.

| 일정 | 출처 | 키(`.env`) |
|---|---|---|
| 국내 종목의 기업설명회(IR)·결산실적 공시 예고·주주총회·배당 기준일 | DART(금융감독원 전자공시) 공시 목록 → 공시 본문에서 날짜 | `DART_API_KEY` 또는 `OPENDART_API_KEY` |
| 내 종목과 이어진 미국 기업의 실적 발표 | Alpha Vantage 실적 일정(한 번 호출로 3개월치, 무료 하루 25번) | `ALPHA_VANTAGE_API_KEY` |
| 미국 경제 지표 발표(CPI·고용·GDP·PCE) | FRED(미국 세인트루이스 연준) 발표 일정 | `FRED_API_KEY` |

- 휴장·미국 휴장·조기 마감·선물옵션 만기일은 웹이 자기 달력(`web/data`의 시장 달력)으로 만든다. 여기서는 모으지 않는다.
- 미국 일정은 **결과가 반영되는 한국 아침 날짜**로 옮긴다(미국 D일 발표 → 한국 D+1일 아침). 원래 버전(`overnight_briefing/service/core/extras.py`, `app/core/data.py`)과 같은 규칙이다.
- 서버 실행: `run_news.py events`, cron 평일 한국 06:00. 키 값은 출력하지 않는다.

## 1. 설정

- `COMPANIES`: 일정을 모을 종목(수집 종목 20개와 같은 목록, `news_collect`의 `COLLECT_COMPANIES`).
- `LOOKBACK_DAYS`: 며칠 전 공시까지 볼지(공시는 보통 일정보다 며칠~몇 주 앞서 나온다).
- `AHEAD_DAYS`: 오늘부터 며칠 뒤 일정까지 넣을지.
- `RESULT_DIR`: 회사 고유번호표·공시 본문 사본을 두는 곳(같은 공시를 다시 받지 않게).

In [ ]:
import datetime as dt
import io
import json
import re
import time
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
import zipfile
from pathlib import Path

import pandas as pd
import sqlalchemy as sa

# ===== 여기만 바꿔서 쓴다 =====
ROOT = Path.cwd()                      # news_search 폴더에서 연다
PROJ = next((p for p in ROOT.parents if (p / '.env').exists()), ROOT.parents[0])   # .env가 있는 폴더
RESULT_DIR = ROOT / 'source' / 'events'
COMPANIES = [            # news_collect의 COLLECT_COMPANIES와 같은 목록(2026-11-23 시연 20종목)
    '000100', '000660', '003490', '005380', '006400', '009540', '010120', '010950', '012450', '017670',
    '042660', '042700', '064350', '068270', '103590', '105560', '222800', '257720', '259960', '278470',
]
LOOKBACK_DAYS = 60
AHEAD_DAYS = 14
RUN = False              # True여야 DB에 넣는다(False면 모은 일정만 보여 준다)
# ==============================

KST = dt.timezone(dt.timedelta(hours=9))
DB_ENV = 'GNEWS_DB_URL'
TERMS_PATH = ROOT / 'source' / 'terms' / 'test_queries.csv'   # 종목코드 → 한국어 이름
assert ROOT.name == 'news_search', 'news_search 폴더에서 열어야 한다.'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
(RESULT_DIR / 'dart_docs').mkdir(exist_ok=True)

## 2. 작은 도구

- `env_value`: `.env`에서 값을 읽는다(값은 출력하지 않는다).
- `get`: 주소를 읽는다. 실패하면 한 번 더 해 보고, 그래도 안 되면 오류를 그대로 올린다.
- `name_key`: 회사 이름 비교용(소문자, 끝의 Inc.·Corp. 등과 문장부호를 뗀다). 종목 이유 노트북과 같다.

In [ ]:
def env_value(*names):
    env_file = PROJ / '.env'
    vals = {}
    if env_file.exists():
        for line in env_file.read_text(encoding='utf-8-sig').splitlines():
            k, _, v = line.partition('=')
            if v.strip():
                vals[k.strip()] = v.strip().strip('"\'')
    return next((vals[n] for n in names if n in vals), None)


def get(url, binary=False, tries=2):
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'overnight-briefing'})
            with urllib.request.urlopen(req, timeout=60) as r:
                data = r.read()
            return data if binary else data.decode('utf-8', 'replace')
        except Exception:
            if k == tries - 1:
                raise
            time.sleep(3)


SUFFIX = re.compile(r'\b(inc|incorporated|corp|corporation|co|company|ltd|limited|plc|llc|holdings?|group|sa|ag|nv|class [a-z])\b')
def name_key(name):
    s = (name or '').lower().replace('&', ' and ')
    s = re.sub(r'^the\s+', '', s)
    s = SUFFIX.sub(' ', re.sub(r'[^\w\s]', ' ', s))
    return re.sub(r'\s+', '', s)


def today_kst():
    return dt.datetime.now(KST).date()


TERMS = pd.read_csv(TERMS_PATH, dtype=str).fillna('')
NAMES = {n.split(':', 1)[1]: c for n, c in TERMS[TERMS['kind'] == '자기 회사'].drop_duplicates('node')[['node', 'company']].values}   # 종목코드 → 이름
print('종목', len(COMPANIES), '/ 이름 없는 종목:', [c for c in COMPANIES if c not in NAMES])

## 3. 국내 공시 일정(DART)

1. 회사 고유번호표(`corpCode.xml`)를 한 번 받아 `source/events/corp_codes.csv`에 우리 종목만 저장한다(있으면 다시 받지 않는다).
2. 종목마다 최근 `LOOKBACK_DAYS`일 공시 목록에서 아래 종류만 고른다(정정 공시는 원 공시 대신 쓴다).
3. 공시 본문(`document.xml`)을 받아(사본은 `source/events/dart_docs/접수번호.txt`) 글자만 남기고, 종류별 머리말 뒤 첫 날짜를 일정 날짜로 쓴다.

| 공시 이름에 들어 있는 말 | 종류 | 날짜를 찾는 머리말 |
|---|---|---|
| 기업설명회(IR) | IR | 일시 |
| 결산실적공시예고 | 실적 발표 예정 | 예정일 · 예정일자 · 공시예정일 |
| 주주총회소집 | 주주총회 | 일시 · 일 시 |
| 주주명부폐쇄 · 배당결정 | 배당 기준일 | 기준일 · 배당기준일 |

- 날짜 모양은 `2026-11-24`, `2026.11.24`, `2026년 11월 24일`을 읽는다. 시각이 `--:--`이면 시각 없이 둔다.
- IR은 본문의 개최목적·대상(예: 국내 주요 기관투자가)을 설명으로 붙인다.

In [ ]:
def corp_codes():
    f = RESULT_DIR / 'corp_codes.csv'
    if f.exists():
        d = pd.read_csv(f, dtype=str)
        if set(COMPANIES) <= set(d['stock_code']):
            return dict(zip(d['stock_code'], d['corp_code']))
    key = env_value('DART_API_KEY', 'OPENDART_API_KEY')
    z = zipfile.ZipFile(io.BytesIO(get(f'https://opendart.fss.or.kr/api/corpCode.xml?crtfc_key={key}', binary=True)))
    root = ET.fromstring(z.read(z.namelist()[0]))
    rows = [{'stock_code': (e.findtext('stock_code') or '').strip(), 'corp_code': e.findtext('corp_code')} for e in root.iter('list')]
    d = pd.DataFrame([r for r in rows if r['stock_code'] in COMPANIES]).drop_duplicates('stock_code')
    d.to_csv(f, index=False)
    return dict(zip(d['stock_code'], d['corp_code']))


KINDS = [   # (공시 이름 정규식, 종류, 날짜 머리말 정규식)
    (r'기업설명회\(IR\)', 'IR', r'일\s*시'),
    (r'결산실적공시\s*예고', '실적 발표 예정', r'(?:공시\s*)?예정\s*일(?:자)?'),
    (r'주주총회소집', '주주총회', r'(?:회의\s*)?일\s*시'),
    (r'주주명부폐쇄|배당결정', '배당 기준일', r'(?:배당\s*)?기준\s*일'),
]
DATE = r'(\d{4})\s*(?:-|\.|년)\s*(\d{1,2})\s*(?:-|\.|월)\s*(\d{1,2})'


def doc_text(rcept_no):
    f = RESULT_DIR / 'dart_docs' / f'{rcept_no}.txt'
    if f.exists():
        return f.read_text(encoding='utf-8')
    key = env_value('DART_API_KEY', 'OPENDART_API_KEY')
    z = zipfile.ZipFile(io.BytesIO(get(f'https://opendart.fss.or.kr/api/document.xml?crtfc_key={key}&rcept_no={rcept_no}', binary=True)))
    raw = ''.join(z.read(n).decode('utf-8', 'replace') for n in z.namelist())
    txt = re.sub(r'\s+', ' ', re.sub(r'<[^>]+>', ' ', raw))
    f.write_text(txt, encoding='utf-8')
    return txt


def first_date(txt, head):
    m = re.search(head + r'[^0-9]{0,40}?' + DATE, txt)
    if not m:
        return None
    try:
        return dt.date(int(m.group(1)), int(m.group(2)), int(m.group(3)))
    except ValueError:
        return None


def after(txt, head, n=80):
    m = re.search(head + r'\s*(.{1,%d}?)\s*(?:\d\.|$)' % n, txt)
    return m.group(1).strip(' :') if m else ''


def dart_events():
    key = env_value('DART_API_KEY', 'OPENDART_API_KEY')
    if not key:
        print('DART 건너뜀: 키가 없다')
        return []
    codes = corp_codes()
    start = today_kst() - dt.timedelta(days=LOOKBACK_DAYS)
    out = []
    for code in COMPANIES:
        if code not in codes:
            print('  고유번호 없음:', code)
            continue
        q = urllib.parse.urlencode(dict(crtfc_key=key, corp_code=codes[code], bgn_de=start.strftime('%Y%m%d'), end_de=today_kst().strftime('%Y%m%d'), page_count=100))
        js = json.loads(get('https://opendart.fss.or.kr/api/list.json?' + q))
        for r in js.get('list', []) or []:
            kind = next(((k, h) for rx, k, h in KINDS if re.search(rx, r['report_nm'])), None)
            if not kind:
                continue
            try:
                txt = doc_text(r['rcept_no'])
            except Exception as e:
                print('  본문 실패', r['rcept_no'], type(e).__name__)
                continue
            d = first_date(txt, kind[1])
            if not d:
                continue
            note = ''
            if kind[0] == 'IR':
                who, why = after(txt, r'참가\s*대상자'), after(txt, r'개최\s*목적')
                note = ' · '.join(x for x in [who, why] if x)
            out.append(dict(date=d, kind=kind[0], codes=[code], title=f"{NAMES.get(code, code)} {kind[0]}", note=note[:160],
                            source='DART', ref=r['rcept_no'], url=f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={r['rcept_no']}",
                            time_kst='', filed=r['rcept_dt'], corrected=r['report_nm'].startswith('[기재정정]')))
        time.sleep(0.2)
    # 같은 종목·종류·날짜는 하나만(정정 공시가 있으면 그것)
    out.sort(key=lambda e: (e['codes'][0], e['kind'], e['date'], not e['corrected'], e['filed']))
    seen, uniq = set(), []
    for e in out:
        k = (e['codes'][0], e['kind'], e['date'])
        if k not in seen:
            seen.add(k)
            uniq.append(e)
    print(f'DART 일정 {len(uniq)}건(공시 {len(out)}건에서)')
    return uniq

## 4. 내 종목과 이어진 미국 기업 실적(Alpha Vantage)

1. 실적 일정(3개월치)을 한 번 받는다.
2. DB 연결표 `news.term_links`에서 우리 종목의 해외 상대(`해외 상대`)를 읽어, 회사 이름(검색어·노드 이름)이 실적 일정의 회사 이름과 맞는 것만 남긴다(이름 비교는 `name_key`, 4글자 이상일 때 앞부분이 같으면 같은 회사).
3. 한국 아침 반영일 = 미국 발표일 + 1일. 미국 장 마감 뒤 발표이고 반영일이 오늘이며 미국 장 마감이 한국 06:00 이후(겨울)면 **06:00 마감 뒤** 일정으로 따로 표시한다(아침 예상(06:30 공개)과 밤사이 브리핑에 들어가지 않는다는 사실만 적는다).

In [ ]:
def us_close_kst(d):
    from zoneinfo import ZoneInfo
    return dt.datetime(d.year, d.month, d.day, 16, tzinfo=ZoneInfo('America/New_York')).astimezone(KST)


def partners(engine):
    with engine.connect() as con:
        rows = con.execute(sa.text("select stock_code, term, node, node_name, name_ko from news.term_links where kind = '해외 상대' and stock_code = any(:c)"),
                           {'c': COMPANIES}).all()
    out = {}
    for code, term, node, node_name, name_ko in rows:
        for nm in {term, node_name, node.split(':', 1)[-1]}:
            k = name_key(nm)
            if len(k) >= 4:
                out.setdefault(k, {'codes': set(), 'name': name_ko or node_name or term})['codes'].add(code)
    return out


def us_earnings(engine):
    key = env_value('ALPHA_VANTAGE_API_KEY')
    if not key:
        print('미국 실적 건너뜀: 키가 없다')
        return []
    f = RESULT_DIR / f'av_earnings_{today_kst():%Y%m%d}.csv'
    if not f.exists():                                   # 하루 한 번만 부른다(무료 하루 25번)
        f.write_text(get('https://www.alphavantage.co/query?' + urllib.parse.urlencode(dict(function='EARNINGS_CALENDAR', horizon='3month', apikey=key))), encoding='utf-8')
    cal = pd.read_csv(f, dtype=str).fillna('')
    if 'symbol' not in cal:
        print('미국 실적 일정을 읽지 못했다:', f.read_text()[:120])
        return []
    pm = partners(engine)
    start, out = today_kst(), []
    for r in cal.itertuples():
        k = name_key(r.name)
        hit = next((v for pk, v in pm.items() if k == pk or k.startswith(pk)), None)
        if not hit or not r.reportDate:
            continue
        us = dt.date.fromisoformat(r.reportDate)
        kr = us + dt.timedelta(days=1)
        hour = {'post-market': '장 마감 뒤', 'pre-market': '장 전'}.get(r.timeOfTheDay, '')
        codes = sorted(hit['codes'])
        tail = ' · 이어진 종목 ' + '·'.join(NAMES.get(c, c) for c in codes[:4]) + (f' 외 {len(codes) - 4}' if len(codes) > 4 else '')
        if kr == start and r.timeOfTheDay == 'post-market' and us_close_kst(us).hour >= 6:
            out.append(dict(date=kr, kind='미국 실적(06:00 마감 뒤)', codes=codes, title=f"{hit['name']} 실적 발표 · 06:00 마감 뒤",
                            note=f'미국 {us.month}/{us.day} 장 마감 뒤(한국 06:00 이후) 발표라 아침 예상 범위(06:30 공개)와 밤사이 브리핑에는 들어가지 않았어요.' + tail,
                            source='Alpha Vantage', ref=r.symbol, url='', time_kst=hour))
            continue
        if not (start < kr <= start + dt.timedelta(days=AHEAD_DAYS)):
            continue
        out.append(dict(date=kr, kind='미국 실적', codes=codes, title=f"{hit['name']} 실적 발표",
                        note=f'미국 {us.month}/{us.day} {hour} 발표 → 한국 {kr.month}/{kr.day} 아침에 반영' + tail,
                        source='Alpha Vantage', ref=r.symbol, url='', time_kst=hour))
    print(f'미국 실적 일정 {len(out)}건(이어진 미국 기업 {len(pm)}곳 기준)')
    return out

## 5. 미국 경제 지표(FRED)

미국 D일 08:30(동부) 발표 → 한국 D일 밤 → **D+1일 아침에 반영**. 내 종목과 상관없이 모두에게 보인다(`codes` 비움).

In [ ]:
FRED_RELEASES = {10: '미국 소비자물가(CPI) 발표', 50: '미국 고용지표 발표', 53: '미국 GDP 발표', 54: '미국 개인소득·소비지출(PCE 물가) 발표'}


def macro_events():
    key = env_value('FRED_API_KEY')
    if not key:
        print('경제 지표 건너뜀: 키가 없다')
        return []
    start, out = today_kst(), []
    for rid, title in FRED_RELEASES.items():
        q = urllib.parse.urlencode(dict(release_id=rid, api_key=key, file_type='json', realtime_start=str(start - dt.timedelta(days=2)),
                                        realtime_end=str(start + dt.timedelta(days=AHEAD_DAYS)), include_release_dates_with_no_data='true'))
        for x in json.loads(get('https://api.stlouisfed.org/fred/release/dates?' + q)).get('release_dates', []):
            us = dt.date.fromisoformat(x['date'])
            kr = us + dt.timedelta(days=1)
            if start < kr <= start + dt.timedelta(days=AHEAD_DAYS):
                out.append(dict(date=kr, kind='미국 경제 지표', codes=[], title=title, note=f'한국 시각 {us.month}/{us.day} 밤 발표 → 다음 날 아침에 반영',
                                source='FRED', ref=str(rid), url='', time_kst=''))
    print(f'경제 지표 일정 {len(out)}건')
    return out

## 6. 합쳐서 DB에 넣기

- 표 `news.company_events`(없으면 만든다): `id`(출처·참조·날짜), `date`(한국 날짜), `kind`, `codes`(이어진 종목코드 목록, 비면 모두에게), `title`, `note`, `source`, `url`, `time_kst`, `created_kst`.
- 매번 이 노트북이 만든 줄(출처 DART·Alpha Vantage·FRED)을 오늘 이후 날짜만 지우고 새로 넣는다(지난 일정은 기록으로 남긴다).
- 웹이 읽을 수 있게 읽기 전용 계정에 읽기 권한을 준다(이미 있으면 그대로).

In [ ]:
T_EVENTS = 'news.company_events'


def save(events, engine):
    now = dt.datetime.now(KST).strftime('%Y-%m-%d %H:%M:%S')
    rows = [dict(id=f"{e['source']}|{e['ref']}|{e['date']}|{','.join(e['codes'])}", date=e['date'], kind=e['kind'], codes=e['codes'], title=e['title'],
                 note=e['note'], source=e['source'], url=e['url'], time_kst=e['time_kst'], created_kst=now) for e in events]
    with engine.begin() as con:
        con.execute(sa.text(f"""create table if not exists {T_EVENTS} (id text primary key, date date not null, kind text, codes text[], title text,
                                note text, source text, url text, time_kst text, created_kst text)"""))
        con.execute(sa.text(f"delete from {T_EVENTS} where source in ('DART', 'Alpha Vantage', 'FRED') and date >= :d"), {'d': today_kst()})
        for r in rows:
            con.execute(sa.text(f"""insert into {T_EVENTS} (id, date, kind, codes, title, note, source, url, time_kst, created_kst)
                                    values (:id, :date, :kind, :codes, :title, :note, :source, :url, :time_kst, :created_kst)
                                    on conflict (id) do update set kind = excluded.kind, codes = excluded.codes, title = excluded.title, note = excluded.note,
                                    url = excluded.url, time_kst = excluded.time_kst, created_kst = excluded.created_kst"""), r)
        for role in con.execute(sa.text("select grantee from information_schema.role_table_grants where table_schema = 'news' and table_name = 'llm_articles' and privilege_type = 'SELECT' and grantee <> current_user")).scalars():
            con.execute(sa.text(f'grant select on {T_EVENTS} to "{role}"'))
    print(f'DB {T_EVENTS}: {len(rows)}줄 넣음')


def collect_events():
    engine = sa.create_engine(env_value(DB_ENV))
    try:
        ev = []
        for f in (dart_events, lambda: us_earnings(engine), macro_events):
            try:
                ev += f()
            except Exception as e:                 # 한 출처가 실패해도 나머지는 넣는다
                print('!!! 일정 출처 실패:', type(e).__name__, str(e)[:200])
        lo, hi = today_kst() - dt.timedelta(days=1), today_kst() + dt.timedelta(days=AHEAD_DAYS)
        ev = [e for e in ev if lo <= e['date'] <= hi]
        ev.sort(key=lambda e: (e['date'], e['kind']))
        display(pd.DataFrame([{k: e[k] for k in ('date', 'kind', 'title', 'note', 'source')} for e in ev]))
        if RUN:
            save(ev, engine)
        else:
            print('RUN = False라 DB에 넣지 않는다.')
        return ev
    finally:
        engine.dispose()

## 7. 실행

In [ ]:
events = collect_events()